# 🧠 تحليل الكود — `BaseIndicator` و `IndicatorResult`

> **ملاحظة سريعة قبل الشرح**: الكود ده هو **حجر الأساس لـ Sprint 3** (Technical Indicators Engine). هو مش أداة مؤشر بحد ذاته، بل **العقد (Contract)** اللي كل المؤشرات المستقبلية هتلتزم بيه.

---

## 📊 المخطط المعماري

```mermaid
graph TD
    A[BaseIndicator ABC] -->|implements| B[SMA]
    A -->|implements| C[EMA]
    A -->|implements| D[RSI]
    A -->|implements| E[MACD]
    
    B --> F[IndicatorResult]
    C --> F
    D --> F
    E --> F
    
    G[list Candle] --> A

    style A fill:#bbf,stroke:#333,stroke-width:3px
    style F fill:#bfb,stroke:#333,stroke-width:2px
```

| الحقل | النوع | الدور |
|-------|------|-------|
| `name` | `str` | اسم المؤشر (`"SMA_20"`, `"RSI_14"`) |
| `timestamp` | `Any` | وقت الحساب (يفضّل `datetime`) |
| `value` | `Decimal \| None` | القيمة الرئيسية — `None` لو غير كافية |
| `extra` | `dict \| None` | قيم إضافية (مثل MACD: `macd`, `signal`, `histogram`) |

**🎯 لماذا `frozen=True`؟** نفس فلسفة `Candle` — **Immutable**، لا تعديل بعد الحساب.

**🎯 لماذا `Decimal` مش `float`؟** نفس مبدأ المشروع — **دقة مالية**.

**🎯 لماذا `value: Decimal | None`؟** مؤشر مثل SMA-20 يحتاج 20 شمعة → لو عندك 10 فقط، القيمة `None` (وليس خطأ).

**🎯 لماذا `extra`؟** مؤشرات مركبة (MACD, Bollinger) تحتاج حقول إضافية.

---

| العنصر | الدور |
|--------|------|
| `ABC` | يمنع إنشاء `BaseIndicator()` مباشرة |
| `@abstractmethod` | يُلزم كل مؤشر بتنفيذ `calculate` |
| `candles: list[Candle]` | مصدر البيانات (يستخدم `Candle` من Sprint 1) |
| `-> IndicatorResult` | نوع الإرجاع موحّد |

---

## ⚠️ ملاحظات على التصميم (اقتراحات لا أخطاء)

| النقطة | الملاحظة |
|--------|----------|
| `timestamp: Any` | الأفضل `datetime` — يستفيد من Type Safety |
| `value: Decimal \| None` | منطقي، لكن `extra` يغطي الحالة لو أردت قيمة موحّدة |
| `pass` بدل `...` | `...` (Ellipsis) أدق للـ ABC |
| لا `logger` | يمكن إضافته في `calculate` للتشخيص |
| لا التحقق من طول `candles` | مسؤولية الـ Implementation (Fail Soft) |

---

## 🧪 اختبار سريع (بدون تنفيذ)

```python
from core.indicator import BaseIndicator, IndicatorResult

# ❌ لن يعمل — ABC
# BaseIndicator()   # TypeError: Can't instantiate abstract class

# ✅ تنفيذ مؤشر مبدئي
class DummyIndicator(BaseIndicator):
    def calculate(self, candles):
        return IndicatorResult(
            name="dummy",
            timestamp=candles[0].timestamp if candles else None,
            value=Decimal("0"),
        )
```

---

## 🗺️ الموقع في خارطة الطريق

```mermaid
graph LR
    A["Sprint 2 ✅<br/>Multi-Pair"] --> B["Sprint 3 🆕<br/>Indicators"]
    B --> B1[core/indicator.py 🆕]
    B --> B2[indicators/sma.py]
    B --> B3[indicators/ema.py]
    B --> B4[indicators/rsi.py]
    B --> B5[indicators/macd.py]
    B --> B6[services/indicator_service.py]

    style B fill:#f9f,stroke:#333,stroke-width:3px
    style B1 fill:#bfb,stroke:#333
```

**الخطوة التالية المنطقية**: بناء **SMA** كأول مؤشر (أبسط واحد) للتحقق من أن العقد شغّال، ثم التوسع لـ EMA/RSI/MACD.

In [4]:
from decimal import Decimal
from indicators.base import BaseIndicator, IndicatorResult # pyright: ignore[reportMissingImports]

# ❌ لن يعمل — ABC
# BaseIndicator()   # TypeError: Can't instantiate abstract class

# ✅ تنفيذ مؤشر مبدئي
class DummyIndicator(BaseIndicator):
    def calculate(self, candles):
        return IndicatorResult(
            name="dummy",
            timestamp=candles[0].timestamp if candles else None,
            value=Decimal("0"),
        )
print(DummyIndicator().calculate([]))  # IndicatorResult(name='dummy', timestamp=None, value=Decimal('0'))

IndicatorResult(name='dummy', timestamp=None, value=Decimal('0'), extra=None)


# 📊 تحليل `SMA` و `EMA`

---

## 🎯 TL;DR

> **SMA** = متوسط بسيط (وزن متساوٍ). **EMA** = متوسط أسّي (وزن أكبر للأحدث). كلاهما يعيد **سلسلة قيم** (واحدة لكل شمعة).

---

## 📊 المخطط

```mermaid
graph TD
    A[BaseIndicator] --> B[SMA]
    A --> C[EMA]
    B --> D[list IndicatorResult]
    C --> D
    E[list Candle] --> B
    E --> C

    style B fill:#bfb,stroke:#333,stroke-width:2px
    style C fill:#bfb,stroke:#333,stroke-width:2px
```

---

## 🔬 SMA — المتوسط البسيط

**القاعدة:** `avg = مجموع إغلاقات آخر N شمعة ÷ N`

**3 حالات في الحلقة:**

| الحالة | الشرط | الناتج |
|--------|-------|--------|
| بيانات غير كافية | `i+1 < period` | `value=None` |
| كافية | `i+1 ≥ period` | `avg` |

**تريكات:**
- `sorted(candles)` → لا نثق بترتيب المدخل.
- `sum(gen, Decimal("0"))` → بدون `Decimal("0")` الناتج `int` وليس `Decimal`.
- `Decimal(str(period))` → تحويل آمن.
- `round(avg, 5)` → 5 خانات فوركس.

---

## 🔬 EMA — المتوسط الأسّي

**القاعدة:** `EMA = close × mult + prev_ema × (1 − mult)`
حيث `mult = 2 / (period + 1)`.

**3 حالات:**

| الحالة | الشرط | الإجراء |
|--------|-------|---------|
| غير كافية | `i+1 < period` | `value=None` |
| **Seed** | `i+1 == period` | `SMA` أول N → `prev_ema` |
| EMA عادي | `i+1 > period` | `close × mult + prev × (1−mult)` |

**تريكات:**
- `multiplier` في `__init__` → حساب مرة واحدة (Performance).
- **Seed بـ SMA** → المعيار المالي (لا يوجد `prev_ema` قبل الشمعة N).
- `prev_ema` state عبر الحلقة.

---

## 📊 مقارنة سريعة

| البند | SMA | EMA |
|-------|-----|-----|
| الوزن | متساوٍ | أكبر للأحدث |
| الاستجابة | بطيئة | سريعة |
| Seed | ❌ | ✅ SMA |
| State | نافذة | `prev_ema` |
| الاستخدام | اتجاه طويل | إشارات سريعة |

---

## ⚠️ ملاحظتان مهمتان

| النقطة | التأثير |
|--------|---------|
| **تعارض العقد** | `BaseIndicator.calculate` يعيد `IndicatorResult` مفرد، لكن SMA/EMA يعيدان `list[IndicatorResult]` → **يجب تعديل العقد** ليعيد `list` |
| **`sorted()` كل استدعاء** | O(N log N) إضافي — مقبول حالياً، يمكن تحسينه لو المدخل مضمون مرتب |

---

## 🧪 اختبار (وصف فقط)

| # | السيناريو | المتوقع |
|---|-----------|---------|
| 1 | SMA(20) + 10 شموع | 10 × `None` |
| 2 | SMA(20) + 25 شمعة | 19 `None` + 6 قيم |
| 3 | EMA(20) + 20 شمعة | 19 `None` + 1 قيمة = SMA |
| 4 | `period < 1` | `ValidationError` |

---

## 🎯 القيمة المعمارية

| المبدأ | التطبيق |
|--------|---------|
| Fail Early | `period < 1` → `ValidationError` |
| Defensive | `sorted()` على المدخل |
| Precision | `Decimal` + `str()` |
| Performance | `multiplier` مرة واحدة |
| Financial Standard | Seed EMA = SMA |

---

## 🗺️ Sprint 3

```mermaid
graph LR
    A["BaseIndicator ✅"] --> B["SMA + EMA 🆕"]
    B --> C["RSI"]
    C --> D["MACD"]
    D --> E["IndicatorService"]

    style B fill:#bfb,stroke:#333,stroke-width:3px
```

In [1]:
from datetime import datetime, timedelta
from decimal import Decimal
from core.candle import Candle
from indicators.moving_averages import SMA, EMA

now = datetime.now()
# بناء 5 شموع متتالية بأسعار إغلاق: 10, 11, 12, 13, 14
candles = [
    Candle(pair='EUR/USD', timestamp=now + timedelta(days=i),
           open=Decimal('10'), high=Decimal('15'), low=Decimal('9'),
           close=Decimal(str(10 + i)))
    for i in range(5)
]

sma3 = SMA(period=3).calculate(candles)
print('--- SMA(3) Results ---')
for r in sma3:
    print(r.timestamp.strftime('%Y-%m-%d'), '->', r.value)

ema3 = EMA(period=3).calculate(candles)
print('\n--- EMA(3) Results ---')
for r in ema3:
    print(r.timestamp.strftime('%Y-%m-%d'), '->', r.value)

--- SMA(3) Results ---
2026-11-05 -> None
2026-11-06 -> None
2026-11-07 -> 11.00000
2026-11-08 -> 12.00000
2026-11-09 -> 13.00000

--- EMA(3) Results ---
2026-11-05 -> None
2026-11-06 -> None
2026-11-07 -> 11.00000
2026-11-08 -> 12.00000
2026-11-09 -> 13.00000


# 📈 تحليل `RSI` و `MACD` — مؤشرات الزخم

---

## 🎯 TL;DR

> **RSI** يقيس قوة الزخم (0-100) عبر مقارنة المكاسب بالخسائر. **MACD** يقيس تقارب/تباعد متوسطين (12 و 26) عبر 3 خطوط: MACD، Signal، Histogram.

---

## 📊 المخطط المعماري

```mermaid
graph TD
    A[BaseIndicator] --> B[RSI]
    A --> C[MACD]
    B --> D[list IndicatorResult]
    C --> E[IndicatorResult + extra]
    F[list Candle] --> B
    F --> C

    style B fill:#bbf,stroke:#333,stroke-width:2px
    style C fill:#bfb,stroke:#333,stroke-width:2px
    style E fill:#f9f,stroke:#333
```

---

## 🔬 RSI — مؤشر القوة النسبية

**القاعدة:** `RSI = 100 - (100 / (1 + RS))`
حيث `RS = avg_gain / avg_loss`.

### 3 مراحل الحساب

```mermaid
graph TD
    A[1. حساب Deltas] --> B[gains/losses arrays]
    B --> C[2. SMA أولي للـ period]
    C --> D[3. Wilder's Smoothing]
    D --> E[RSI لكل شمعة]

    style C fill:#f9f,stroke:#333,stroke-width:2px
    style D fill:#bfb,stroke:#333,stroke-width:2px
```

### المرحلة 1: الفروقات

| الحالة | Gains | Losses |
|--------|-------|--------|
| `delta > 0` | `delta` | `0` |
| `delta ≤ 0` | `0` | `abs(delta)` |

**🎯 تريكة**: `gains[0] = losses[0] = 0` — لا يوجد فرق قبل أول شمعة.

### المرحلة 2: المتوسط الأولي (SMA)

```python
first_avg_gain = sum(gains[1:period+1]) / period
first_avg_loss = sum(losses[1:period+1]) / period
```

**لماذا `[1:period+1]`؟** لأن `gains[0]` = 0 (لا فرق)، فنأخذ `period` فرقاً فعلياً.

### المرحلة 3: Wilder's Smoothing

```python
avg_gain = (avg_gain × (period-1) + gain[i]) / period
```

**🎯 الفرق عن EMA**: Wilder يستخدم `(period-1)/period` بدلاً من `2/(period+1)` — **أبطأ استجابة**، **أكثر ثباتاً**.

### معالجة الحالات الحدية

| الحالة | الناتج | السبب |
|--------|--------|-------|
| `avg_loss == 0` | `100.00` | كل الشموع صاعدة (ذروة شراء) |
| `avg_gain == 0` | `0.00` | كل الشموع هابطة (ذروة بيع) |

**🎯 تريكة الأمان**: تجنب `DivisionByZero`.

### التوقيت (Timing)

| الشمعة | RSI |
|--------|-----|
| `0..period-1` | `None` |
| `period` | أول قيمة (من SMA) |
| `period+1..n-1` | Wilder's smoothing |

---

## 🔬 MACD — تقارب/تباعد المتوسطات

**الفكرة:** `MACD = EMA(12) - EMA(26)` — يقيس الفرق بين متوسط سريع وبطيء.

### 4 مراحل

```mermaid
graph TD
    A[1. Fast EMA 12] --> D[2. MACD = Fast - Slow]
    B[1. Slow EMA 26] --> D
    D --> E[3. Signal = EMA 9 of MACD]
    E --> F[4. Histogram = MACD - Signal]

    style D fill:#bbf,stroke:#333,stroke-width:2px
    style E fill:#f9f,stroke:#333,stroke-width:2px
    style F fill:#bfb,stroke:#333,stroke-width:2px
```

### الخطوط الثلاثة

| الخط | الحساب | المعنى |
|------|--------|--------|
| **MACD** | `EMA(12) - EMA(26)` | الاتجاه |
| **Signal** | `EMA(9) of MACD` | تأكيد الاتجاه |
| **Histogram** | `MACD - Signal` | قوة الزخم |

### التعامل مع `None` — التحدي الأكبر

```mermaid
graph LR
    A[fast_ema: list Decimal None] --> C[zip]
    B[slow_ema: list Decimal None] --> C
    C --> D{كلاهما not None?}
    D -->|Yes| E[macd = f - s]
    D -->|No| F[macd = None]

    style E fill:#bfb,stroke:#333
    style F fill:#fdd,stroke:#333
```

**المشكلة:** أول `slow_period - 1` شمعة ستكون `None` في `slow_ema`.

**الحل:** `zip` + فحص `is not None` قبل الطرح.

### 🎯 التريكة الأصعب: `_calculate_ema_series_from_nullable`

**السؤال:** كيف نحسب EMA على سلسلة **تحتوي None في البداية**؟

**الحل الذكي:**

```mermaid
graph TD
    A[series: list Decimal None] --> B[valid_indices = [idx where v not None]]
    B --> C{len valid >= period?}
    C -->|No| Z[return all None]
    C -->|Yes| D[start_idx = valid_indices[period-1]]
    D --> E[SMA أولي على أول period valid]
    E --> F[prev_ema = SMA]
    F --> G[Loop على valid_indices[period:]]
    G --> H[EMA = val*mult + prev*1-mult]

    style B fill:#bbf,stroke:#333,stroke-width:2px
    style E fill:#f9f,stroke:#333
    style H fill:#bfb,stroke:#333
```

**الفكرة:** 
1. استخراج **مواقع** القيم الفعلية (`valid_indices`).
2. أول `period` قيمة صالحة → SMA → **Seed**.
3. الباقي → EMA عادي.

**🎯 النتيجة:** `Signal` يبدأ **بعد** `MACD` بمقدار `signal_period - 1` شمعة.

### استخراج `extra`

```python
extra = {
    "macd": round(m_val, 5),
    "signal": round(s_val, 5),
    "histogram": round(hist_val, 5),
}
```

**`value = extra["macd"]`** — القيمة الأساسية للـ `IndicatorResult` هي خط MACD.

---

## 📊 مقارنة RSI vs MACD

| البند | RSI | MACD |
|-------|-----|------|
| النوع | زخم (Momentum) | تقارب/تباعد |
| المدى | 0-100 | لا محدود |
| عدد الخطوط | 1 | 3 (MACD/Signal/Hist) |
| متوسطات | Wilder's | EMA |
| التمهيد | مرة واحدة | 3 طبقات (12, 26, 9) |
| التوقيت الأول | `period` | `slow_period + signal_period - 1` |

---

## ⚠️ ملاحظات على التصميم

| النقطة | التأثير |
|--------|---------|
| **`assert prev_ema is not None`** | يعمل في dev، قد يُتجاهل في `python -O` — الأفضل `if prev_ema is None: raise` |
| **`round(..., 5)` للـ MACD** | يخسر دقة — لكن مقبول للعرض |
| **`_calculate_ema_series_from_nullable`** | منطق معقد — يمكن استخراجه لـ utility |
| **`extra` في كل `IndicatorResult`** | ذاكرة إضافية — لكن مرونة عالية |

---

## 🧪 اختبار (وصف)

| # | السيناريو | المتوقع |
|---|-----------|---------|
| 1 | RSI(14) + 10 شموع | 10 × `None` |
| 2 | RSI(14) + 20 شمعة | 14 `None` + 6 قيم (0-100) |
| 3 | RSI + كل الشموع صاعدة | `100.00` |
| 4 | RSI + كل الشموع هابطة | `0.00` |
| 5 | MACD + 30 شمعة | `extra` كامل |
| 6 | `fast_period >= slow_period` | `ValidationError` |

---

## 🎯 القيمة المعمارية

| المبدأ | التطبيق |
|--------|---------|
| Fail Early | `period < 1` → `ValidationError` |
| Financial Standard | Wilder's Smoothing |
| Precision | `Decimal` في كل خطوة |
| Edge Cases | معالجة 0/0 |
| Nullable Handling | `valid_indices` للتخطي الذكي |
| Multi-output | `extra` dict للخطوط الثلاثة |

---

## 🗺️ Sprint 3

```mermaid
graph LR
    A["BaseIndicator ✅"] --> B["SMA + EMA ✅"]
    B --> C["RSI + MACD 🆕"]
    C --> D["IndicatorService"]
    D --> E["CLI: forex indicator"]

    style C fill:#bfb,stroke:#333,stroke-width:3px
    style D fill:#ddd,stroke:#333
```

In [2]:

from datetime import datetime, timedelta
from decimal import Decimal
from core.candle import Candle
from indicators.momentum import RSI, MACD

now = datetime.now()
# محاكاة 35 شمعة متصاعدة ومتذبذبة
candles = [
    Candle(
        pair='EUR/USD',
        timestamp=now + timedelta(days=i),
        open=Decimal('1.0000'), high=Decimal('1.1000'), low=Decimal('0.9500'),
        close=Decimal(str(1.0500 + (i * 0.0020 if i % 2 == 0 else -0.0010)))
    )
    for i in range(35)
]

rsi_results = RSI(period=14).calculate(candles)
macd_results = MACD(fast_period=12, slow_period=26, signal_period=9).calculate(candles)

print('--- RSI Last 3 Values ---')
for r in rsi_results[-3:]:
    print(r.timestamp.strftime('%Y-%m-%d'), '-> RSI:', r.value)

print('\n--- MACD Last 3 Values ---')
for r in macd_results[-3:]:
    print(r.timestamp.strftime('%Y-%m-%d'), '->', r.extra)


--- RSI Last 3 Values ---
2026-12-07 -> RSI: 54.09
2026-12-08 -> RSI: 48.26
2026-12-09 -> RSI: 53.93

--- MACD Last 3 Values ---
2026-12-07 -> {'macd': Decimal('0.00820'), 'signal': None, 'histogram': None}
2026-12-08 -> {'macd': Decimal('0.00522'), 'signal': Decimal('0.00650'), 'histogram': Decimal('-0.00128')}
2026-12-09 -> {'macd': Decimal('0.00833'), 'signal': Decimal('0.00687'), 'histogram': Decimal('0.00146')}


# 🔬 تحليل `analyze_pair` — طبقة التحليل الفني

---

## 🎯 TL;DR

> دالة `analyze_pair` هي **نقطة الدخول للتحليل الفني**: تجلب الشموع من DB، تُمرّرها على المؤشرات المطلوبة، وتُعيد dict موحّداً بالنتائج.

---

## 📊 مخطط التدفق

```mermaid
graph TD
    A[analyze_pair] --> B[1. CurrencyPair.from_string]
    B --> C[2. Interval.from_str]
    C --> D[3. storage.get_candles]
    D --> E{candles empty?}
    E -->|Yes| F[return empty result]
    E -->|No| G[4. sorted ascending]
    G --> H[5. parse indicators list]
    H --> I{which indicators?}
    I -->|sma| J[SMA 20]
    I -->|ema| K[EMA 20]
    I -->|rsi| L[RSI 14]
    I -->|macd| M[MACD 12/26/9]
    J --> N[6. return dict]
    K --> N
    L --> N
    M --> N

    style D fill:#bbf,stroke:#333,stroke-width:2px
    style N fill:#bfb,stroke:#333,stroke-width:2px
```

---

## 🔬 التشريح — 6 خطوات

### 1️⃣ التحقق المزدوج

```
CurrencyPair.from_string(pair)   → توحيد الزوج
Interval.from_str(interval)      → توحيد الإطار
```

**Fail Early** قبل أي I/O.

---

### 2️⃣ جلب الشموع من DB

```
storage.get_candles(pair, interval, limit=50)
```

**🎯 لماذا `limit=50` افتراضياً؟**
- SMA(20) و RSI(14) يحتاجان 20-14 شمعة على الأقل.
- MACD يحتاج `26 + 9 - 1 = 34` شمعة للـ Signal.
- **50** = هامش أمان لجميع المؤشرات.

**🎯 لماذا من DB وليس API؟**
- التحليل الفني يحتاج **تاريخاً**، والـ API يجلب نقطة واحدة.
- DB أسرع ولا يستهلك quota.

---

### 3️⃣ فحص الفراغ

```
if not candles → return {pair, candles_count: 0, results: {}}
```

**Fail Soft**: لا نرمي exception — نُعيد هيكلاً فارغاً.

---

### 4️⃣ الفرز التصاعدي

```
candles_sorted = sorted(candles, key=lambda c: c.timestamp)
```

**🎯 لماذا فرز إضافي؟**
- `get_candles` يُرجع بـ `ORDER BY timestamp DESC` (الأحدث أولاً).
- المؤشرات تحتاج ترتيباً **تصاعدياً** (الأقدم أولاً) للحساب الصحيح.

---

### 5️⃣ المؤشرات المطلوبة

```
requested_indicators = [ind.strip().lower() for ind in (indicators or ["sma", "rsi", "macd"])]
```

| الميزة | التفصيل |
|--------|---------|
| **Default** | `["sma", "rsi", "macd"]` — 3 مؤشرات شائعة |
| **`.strip().lower()`** | تطبيع المدخل (`" SMA "` → `"sma"`) |
| **`in` check** | `if "sma" in requested` |

**🎯 لماذا `ema` غير موجود في الـ Default؟**
- لأنه مشابه لـ SMA في الاستخدام — يُطلب صراحة عند الحاجة.

---

### 6️⃣ بناء النتائج

```
indicator_outputs = {
    "sma_20": [IndicatorResult...],
    "rsi_14": [IndicatorResult...],
    "macd":   [IndicatorResult...],
}
```

**هيكل الإرجاع النهائي:**

| المفتاح | القيمة |
|---------|--------|
| `pair` | `"EUR/USD"` |
| `interval` | `"1day"` |
| `candles_count` | `len(candles_sorted)` |
| `latest_candle` | `Candle` (الأحدث) |
| `indicators` | `dict[name, list[IndicatorResult]]` |

---

## ⚠️ ملاحظات على التصميم

| النقطة | الملاحظة | التأثير |
|--------|----------|---------|
| **`SMA(period=20)` ثابت** | لا يمكن تمرير `period` مخصص | محدودية |
| **`indicators or [...]`** | لو `indicators=[]` → يستخدم الـ default | **سلوك غير متوقع**: `[]` = كل الافتراضيات |
| **`if "sma" in requested`** | يفتقد `sma_50` مثلاً | لا يدعم مؤشرات متعددة بنفس النوع |
| **لا `try/except` حول المؤشرات** | لو مؤشر فشل → الدالة كلها تفشل | **يجب** عزل الأخطاء |
| **`latest_candle` كائن كامل** | لا يمكن serialize بسهولة | يُفضّل `to_dict()` |
| **لا `logger.info`** | لا تتبع | صعوبة الديباج |

---

## 🧪 اختبار (وصف)

| # | السيناريو | المتوقع |
|---|-----------|---------|
| 1 | `pair=EUR/USD, limit=50` | dict كامل بـ 3 مؤشرات |
| 2 | `indicators=["rsi"]` | فقط `rsi_14` |
| 3 | `indicators=["ema"]` | فقط `ema_20` |
| 4 | DB فاضي | `{candles_count: 0, results: {}}` |
| 5 | `pair="INVALID"` | `ValidationError` |
| 6 | `interval="3days"` | `ValueError` |

---

## 🎯 القيمة المعمارية

| المبدأ | التطبيق |
|--------|---------|
| Fail Early | `CurrencyPair` + `Interval` قبل DB |
| Fail Soft | DB فاضي → dict فارغ |
| Reuse | نفس `CurrencyPair`/`Interval` من Sprint 2 |
| Separation | التحليل من DB (ليس API) |
| Sorted Input | ضمان ترتيب تصاعدي للمؤشرات |
| Flexible Output | `dict` مرن مع `indicators` customizable |

---

## 🗺️ Sprint 3

```mermaid
graph LR
    A["SMA + EMA ✅"] --> B["RSI + MACD ✅"]
    B --> C["ForexService.analyze_pair 🆕"]
    C --> D["CLI: forex indicator"]
    D --> E["tests/test_indicators.py"]

    style C fill:#bfb,stroke:#333,stroke-width:3px
    style D fill:#ddd,stroke:#333
```

---

## 📋 الخطوة التالية

> **CLI**: أمر `forex indicator --pair EUR/USD --interval 1day --indicators sma,rsi,macd`

**ما سيُضاف:**
- `cmd_indicator` في `commands.py` → يستدعي `analyze_pair` ويطبع جدولاً.
- `add_parser("indicator")` في `parser.py` → `--pair`, `--interval`, `--indicators`.

**السؤال الجوهري:** كيف نعرض **سلسلة قيم** (50 شمعة × 3 مؤشرات) في Terminal؟

**الجواب:** نطبع **آخر N نتائج** فقط (مثلاً 5) + `latest` value لكل مؤشر. باقي السلسلة للاستخدام البرمجي/الشـارت.

In [3]:

from services.forex_service import ForexService

service = ForexService()
# التأكد من وجود بيانات للتحليل عبر جلب 30 شمعة
service.fetch_and_store(pair='EUR/USD', interval='1day', limit=30)

res = service.analyze_pair(pair='EUR/USD', interval='1day', limit=30, indicators=['sma', 'rsi', 'macd'])

print('Analysis Completed for:', res['pair'])
print('Latest Close:', res['latest_candle'].close)
if 'rsi_14' in res['indicators']:
    latest_rsi = res['indicators']['rsi_14'][-1].value
    print('Latest RSI(14):', latest_rsi)
if 'macd' in res['indicators']:
    latest_macd = res['indicators']['macd'][-1].extra
    print('Latest MACD:', latest_macd)

[2026-11-05 00:24:00] [INFO] [forex_tracker]: Connected to SQLite database at: ./data\candles.db
[2026-11-05 00:24:00] [INFO] [forex_tracker]: جلب وتخزين 30 شمعة لزوج EUR/USD (1day)...
[2026-11-05 00:24:00] [INFO] [forex_tracker]: جلب 30 شمعة لزوج EUR/USD بإطار 1day من TwelveData...
[2026-11-05 00:24:00] [INFO] [forex_tracker]: إرسال طلب HTTP GET إلى: https://api.twelvedata.com/time_series
[2026-11-05 00:24:02] [INFO] [forex_tracker]: تم بنجاح استخراج 30 شمعة لزوج EUR/USD.
[2026-11-05 00:24:02] [INFO] [forex_tracker]: Saved 30 candle(s) to SQLite.


Analysis Completed for: EUR/USD
Latest Close: 1.12224
Latest RSI(14): 52.40
Latest MACD: {'macd': Decimal('-0.00090'), 'signal': None, 'histogram': None}
